# 第 1 章 · 初识 Google ADK：十分钟跑通第一个 Agent

> 本章目标：
> 1. 理解 ADK 的定位与五大核心概念；
> 2. 用 **DeepSeek** 模型跑通你的第一个 ADK Agent；
> 3. 学会"解剖"一次 Agent 运行——看懂事件流（Event Stream）。

---

## 1. ADK 是什么？

**ADK（Agent Development Kit）** 是 Google 开源的 Python 框架，用于构建、评估和部署 AI 智能体。它有几个鲜明的性格：

| 特质 | 说明 |
|---|---|
| 🧬 **Google 血统，模型中立** | 与 Gemini / Vertex AI 深度集成，但通过内置 **LiteLLM** 适配层可接入 DeepSeek、OpenAI、Claude 等 100+ 模型 |
| 🤖 **Agent 一等公民** | 一切围绕 `Agent` 抽象展开，多智能体层级（`sub_agents`）是语言内建概念，而非后期补丁 |
| 🎛️ **开发体验优先** | 自带 `adk web` 可视化调试界面、`adk run` 命令行工具，事件流完全透明 |
| 🚀 **面向生产** | 内置评估框架，可一键部署到 Vertex AI Agent Engine 或 Cloud Run |

```mermaid
flowchart TB
    subgraph DEV["开发态"]
        CODE["你的 Python 代码<br/>Agent + Tools"] --> CLI["adk web / adk run<br/>本地调试"]
    end
    subgraph CORE["ADK 运行时核心"]
        AGT["🤖 Agent<br/>（LLM 驱动的决策者）"]
        RUN["🏃 Runner<br/>（执行引擎）"]
        SES["💾 Session Service<br/>（会话与状态）"]
        MEM["🧠 Memory Service<br/>（长期记忆）"]
        RUN --> AGT
        RUN --> SES
        RUN --> MEM
    end
    subgraph PROD["生产态"]
        AE["Vertex AI Agent Engine"]
        CR["Cloud Run / 任意容器"]
    end
    CLI --> RUN
    CORE --> PROD
    style AGT fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style RUN fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

---

## 2. 五大核心概念（先混个脸熟）

| 概念 | 类比 | 作用 |
|---|---|---|
| **Agent** | 一位员工 | 拥有指令（instruction）、模型和工具，负责做决策 |
| **Tool** | 员工的技能/装备 | 函数、搜索、甚至另一个 Agent，供 Agent 调用 |
| **Session** | 一次完整的工单 | 一次对话的完整上下文（历史消息 + 状态） |
| **Event** | 工单上的一条条记录 | Agent 运行中产生的每一个动作（发言、调工具、转移） |
| **Runner** | 项目经理 | 驱动整个"思考→行动"循环，把事件流回给你 |

它们的关系用一张图概括：

```mermaid
sequenceDiagram
    participant U as 用户
    participant R as Runner
    participant A as Agent
    participant T as Tool
    U->>R: 新消息 (Content)
    R->>A: 携带 Session 上下文调用
    A->>A: LLM 推理
    A-->>R: Event: 请求调用工具
    R->>T: 执行工具函数
    T-->>R: 工具结果
    R->>A: 把结果送回 LLM
    A-->>R: Event: 最终文本回复
    R-->>U: 逐条 yield Event
```

> 📌 **关键直觉**：ADK 的运行结果不是"一个字符串"，而是**一串事件（Event）**。理解事件流，就理解了 ADK 的一半。

---

## 3. 环境准备

```bash
pip install google-adk litellm   # 本教程环境已装好
```

ADK 原生默认走 Gemini（需要 `GOOGLE_API_KEY` 或 Vertex AI 凭证）。本教程使用 **DeepSeek**，通过 LiteLLM 适配层接入——LiteLLM 原生支持 DeepSeek，约定：

- 模型名写成 `deepseek/deepseek-chat`（`deepseek/` 是 LiteLLM 的 provider 前缀）；
- 密钥从环境变量 `DEEPSEEK_API_KEY` 自动读取。

先做环境自检：


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

import importlib.metadata as im
print("google-adk:", im.version("google-adk"))
print("litellm   :", im.version("litellm"))
print("✅ 环境就绪")


google-adk: 2.6.3
litellm   : 1.96.2
✅ 环境就绪


---

## 4. 第一个 ADK Agent

下面创建一个最小的 Agent——一个"中文小助手"，并用 **Runner** 驱动它回答一个问题。

三个关键步骤：
1. `LiteLlm(model="deepseek/deepseek-chat")` —— 声明底层模型；
2. `Agent(...)` —— 声明智能体（名字 + 模型 + 指令）；
3. `Runner` + `SessionService` —— 提供运行上下文并驱动执行。


In [2]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

# 1. 定义模型：通过 LiteLLM 适配层接入 DeepSeek
deepseek_model = LiteLlm(model="deepseek/deepseek-chat")

# 2. 定义 Agent：名字、模型、指令（system prompt）
assistant = Agent(
    name="chinese_helper",
    model=deepseek_model,
    instruction="你是一个热情的中文学习助手，回答简洁，适当使用 emoji。",
    description="一个中文学习小助手",
)

print("Agent 创建成功：", assistant.name)


Agent 创建成功： chinese_helper


接下来是 ADK 特有的两个"基础设施"：

- **`InMemorySessionService`**：内存版会话服务（生产中可换成数据库实现）；
- **`Runner`**：执行引擎，`run_async()` 返回一个**异步事件迭代器**。

> 💡 Jupyter 单元格原生支持顶层 `await`，所以我们可以直接写异步代码，无需 `asyncio.run()`。


In [3]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "adk_tutorial"
USER_ID = "student_01"

# 3. 组装运行时
session_service = InMemorySessionService()
# 使用 Runner 加载 InMemorySessionService 和 `Agent 进行基础的回话.
runner = Runner(agent=assistant, app_name=APP_NAME, session_service=session_service)

async def chat(query: str, session_id: str = "s1") -> str:
    """向 Agent 发送一条消息，收集最终回复文本。"""
    # 确保会话存在（重复创建同 id 会话会报错，做个防御）
    try:
        await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    except Exception:
        pass

    # 输入 message
    message = types.Content(role="user", parts=[types.Part(text=query)])
    final_text = ""

    # 遍历 event 直到 event.is_final_response()
    async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=message):
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text
    return final_text

reply = await chat("你好！请用一句话解释什么是 AI Agent。")
print(reply)


00:24:06 - LiteLLM:WARNING: get_model_cost_map.py:289 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000). Falling back to local backup.


你好！😊 AI Agent 就是一个能自己感知环境、做出决策并采取行动来完成任务的智能程序。🤖


🎉 恭喜，第一个 ADK Agent 已经跑通！

但刚才我们只取了"最终回复"。ADK 真正的世界观是**事件流**——下一节把它拆开看。

---

## 5. 解剖一次运行：Event 流

`runner.run_async()` 每 `yield` 一个 `Event`，代表运行中的一个动作。每个 Event 携带：

| 字段 | 含义 |
|---|---|
| `author` | 事件来源（Agent 名 / `user`） |
| `content` | 内容（文本、函数调用、函数响应） |
| `actions` | 副作用（状态变更、Agent 转移、 escalate 等） |
| `is_final_response()` | 是否是本轮的最终答复 |

我们换个会触发"多轮内心戏"的问题，把事件逐个打印出来：


In [4]:
async def chat_verbose(query: str, session_id: str = "s2"):
    try:
        await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    except Exception:
        pass
    message = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 用户: {query}\n" + "─" * 40)
    idx = 0
    async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=message):
        idx += 1
        kind = []
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text: kind.append(f"文本[{p.text[:50]}...]" if len(p.text) > 50 else f"文本[{p.text}]")
                if p.function_call: kind.append(f"调用工具[{p.function_call.name}]")
                if p.function_response: kind.append(f"工具返回[{p.function_response.name}]")
        print(f"Event#{idx} | author={event.author} | final={event.is_final_response()} | {'; '.join(kind) or '(无内容)'}")

await chat_verbose("给我出一个关于成语守株待兔的选择题，并公布答案。")


🧑 用户: 给我出一个关于成语守株待兔的选择题，并公布答案。
────────────────────────────────────────
Event#1 | author=chinese_helper | final=True | 文本[好的，来看这道题！📚

**题目：**

成语“守株待兔”比喻的是什么？

A. 勤劳耕作，等待丰收...]


可以看到：即使是简单对话，事件流也精确记录了"谁、在什么阶段、产出了什么"。当 Agent 开始调用工具、进行多智能体转移时（第 3、4 章），事件流就是你最重要的调试窗口。

> 🔍 **补充**：`adk web` 命令可以启动一个本地 Web UI，把上面这个事件流渲染成交互式时间轴，还能查看 Session 状态。在终端运行 `adk web 你的agent目录` 即可体验（需在工程目录结构下使用，本教程聚焦 API 层）。

---

## 6. 多轮对话：Session 让 Agent 拥有"记忆"

同一个 `session_id` 下的多次调用共享对话历史——这就是 Session 的第一重价值：


In [5]:
r1 = await chat("记住：我最喜欢的颜色是蓝色。", session_id="s3")
print("第1轮：", r1)

r2 = await chat("我最喜欢的颜色是什么？用它造一个句子。", session_id="s3")
print("第2轮：", r2)


第1轮： 好的，我记住了！你最喜欢的颜色是蓝色 💙。
第2轮： 你最喜欢的颜色是蓝色 💙。

造句：我最喜欢的颜色是蓝色，因为它让我想起广阔的天空和宁静的大海 🌊。


第二问能答出"蓝色"，说明 Session 中保存了完整历史。第 5 章会深入 Session / State / Memory 三层记忆体系。

---

## 7. 与 LangChain 对照 🔄

| 你刚做的事 | ADK 写法 | LangChain 对应物 |
|---|---|---|
| 定义模型 | `LiteLlm(model="deepseek/deepseek-chat")` | `ChatOpenAI(model=..., base_url=...)` |
| 定义"带人设的模型" | `Agent(instruction=...)` | `ChatPromptTemplate` + LLM，或 `create_agent` |
| 跑一次对话 | `Runner.run_async()` 事件流 | `llm.invoke()` / `agent.invoke()` 返回完整结果 |
| 多轮记忆 | `SessionService`（框架内建） | LangGraph `Checkpointer` 或手工传 messages |

**哲学差异初显**：ADK 把"会话管理"内建在框架里（Session 是必需品）；LangChain 基础层是无状态的（记忆是可选项，LangGraph 用 Checkpointer 补上）。

---

## 📌 本章要点回顾

- ADK 五大概念：**Agent / Tool / Session / Event / Runner**；
- 通过 `LiteLlm(model="deepseek/deepseek-chat")` + `DEEPSEEK_API_KEY` 即可让 ADK 跑在 DeepSeek 上；
- `run_async()` 产出的是**事件流**，`is_final_response()` 标记最终答复；
- 同一 `session_id` 下自动拥有多轮记忆。

> ➡️ 下一章：[02-Agent详解与模型接入](02-Agent详解与模型接入.ipynb) —— 把 `Agent` 的每个参数、每种玩法讲透。


---

## 🧪 本章练习

### 1. 改造第一个 Agent（基础）

把示例 Agent 改造成“学习计划助手”：通过 `instruction` 约束它先追问目标与可用时间，再给出 Markdown 表格计划。分别输入信息充分和信息不足的请求，确认回复行为符合约束；同时记录模型名、Agent 名称及最终响应。

### 2. 观察而不是只看答案（基础）

为一次运行编写事件观察器，逐条打印事件作者、是否为最终响应以及事件携带的内容类型，最后只汇总最终文本。目标是能从输出中解释 `Runner`、`Event` 与 Agent 回复之间的关系，而不是简单调用后打印结果。

### 3. 验证会话隔离（进阶）

创建两个用户、至少三个 `session_id`，让它们分别记住不同的颜色或项目代号，再交叉提问。写出自动化断言，证明同一 Session 能延续上下文、不同 Session 不会串话；补充说明 `user_id` 与 `session_id` 各自承担什么职责。

### 4. 封装一个可复用的对话入口（工程）

把本章代码整理为一个命令行小程序或独立 Python 模块：首次访问时创建 Session，后续复用；支持退出命令；API Key 缺失、模型请求失败时给出清晰错误；业务代码不能散落全局变量。验收效果是连续对话可用，重新选择新 Session 后上下文被隔离。

### 5. 跨框架预习（思考）

根据第 0 章的概念映射，推测上述程序若改用 LangChain/LangGraph，`Agent`、`Runner`、`Event` 和 `SessionService` 分别可能由什么承担。暂时无需编码，但要标出你最不确定的一处，并在学习另一条线路后回来修订。
